#### Preamble for `CoLab`

To use this notebook (if you haven't already) you can first save a copy to your local drive by clicking `File > Save a Copy in Drive` and run on that copy.

_Note_: `Colab` is a really handy way to test and try `strauss`, though it will generally run and display audio more slowly than running on your local machine. There are also RAM limitations that limit the size and complexity of sonifications we can make. For a more responsive experience, why not install `strauss` locally, following the instructions [on the Github](https://github.com/james-trayford/strauss)?

Run these cells, so that the notebook functions on the _Google_ `Colab` platform:

In [ ]:
!git clone -b animation_v2 --single-branch https://github.com/james-trayford/strauss.git
%pip --quiet install "strauss @ git+https://github.com/james-trayford/strauss.git@animation_v2" -q

In [ ]:
#!git clone https://github.com/james-trayford/strauss.git
#%pip --quiet install strauss

In [ ]:
%cd strauss/examples/

### <u> Generate the sunrise to sunset sonification used in the "_Audible Universe_" planetarium show </u>

**First, import relevant modules:**

In [ ]:
import matplotlib.pyplot as plt
import wavio as wav
from strauss.sonification import Sonification
from strauss.sources import Objects
from strauss import channels
from strauss.score import Score
import numpy as np
from strauss.generator import Sampler
import IPython.display as ipd
import glob
import os
import copy
%matplotlib inline

First we download the samples to the local data directory, if they haven't been already:

In [ ]:
outdir = "../data/samples/day_sequence/"

if glob.glob(f"{outdir}/*.wav"):
    print(f"Directory {outdir} already exists.")
else:
    print("Downloading files...")
    import urllib.request
    import os

    path = os.path.realpath('../data/samples/day_sequence/')
    os.mkdir(path)

    files = ("sun_A4.wav", "scatter_B4.wav")
    urls = ("https://drive.google.com/uc?export=download&id=15D7xHEKtKppTvzzwECIq_0UGhifdhrEy",
            "https://drive.google.com/uc?export=download&id=1bnhZ_kagtWMUkj1VtEE6vzQGfnYexQfL")
    for f, u in zip(files, urls):
        with urllib.request.urlopen(u) as response, open(f"{path}/{f}", 'wb') as out_file:
            print(f"\t getting {f}")
            data = response.read() # a `bytes` object
            out_file.write(data)
    print("Done.")

**Specify the audio system to use** _(use `'stereo'` by default but for the planetarium `'5.1'` is used)_

In [ ]:
# specify audio system (e.g. mono, stereo, 5.1, ...)
system = "stereo"

**Now, set-up the sampler:**

In [ ]:
# set up sampler
sampler = Sampler("../data/samples/day_sequence")
sampler.modify_preset({'filter':'on'}) # want filtering on for sun altitude effect

**Set mapping limits of mapped quantities**

In [ ]:
maplims =  {'azimuth': (0, 360),
            'polar': (0, 180),
            'pitch' : (0, 1),
            'cutoff' : (0, 1),
            'volume' : (0,1),
            'time_evo' : (0,147)}

**Initialise the score:**

In [ ]:
# setup score
score =  Score([['A4','B4']], 120)

**Render sonification for specified planet...**

In [ ]:
data = {'azimuth': np.array([90,90, 0, 330, 240,240]),
        'polar': np.array([45,45,0, 40, 0, 0]), # constant polar of 90 deg
        'pitch': 1,     # constant pitch
        'volume': np.ones(6),
        'cutoff': np.array([0.5, 0.5, 1, 0.444, 0, 0]),
        'time_evo': np.array([0, 33.5,45, 57.5, 72.5, 147])}

# set up source
events = Objects(maplims.keys())
events.fromdict(data)
events.apply_mapping_functions(map_lims=maplims)

soni = Sonification(score, events, sampler, system)
soni.render()

**Listen to and plot the waveforms from the sonification:**

In [ ]:
soni.notebook_display()

In [ ]:
data2 = {'azimuth': np.ones(8)*0,
        'polar': np.zeros(8), # constant polar of 90 deg
        'pitch': 1,     # constant pitch
        'volume': np.array([0.2,0.2,0.4,0.2,0.1,0.03, 0.01, 0.]),
        'cutoff': np.ones(8),
        'time_evo': np.array([0, 33.5,45, 57.5, 72.5, 90, 100, 147])}

# set up source
events2 = Objects(maplims.keys())
events2.fromdict(data2)
events2.apply_mapping_functions(map_lims=maplims)

sampler2 = copy.deepcopy(sampler)
sampler2.samples['A4'] = sampler2.samples['B4']

soni2 = Sonification(score, events2, sampler2, system)
soni2.out_channels = soni.out_channels
soni2.render()

In [ ]:
soni2.notebook_display()

**Combine and save sonification to a multi-channel wav**

NOTE: Change `"../../FILENAME.wav"` to your filepath of choice. By default, the sound file is normalised to that of the highest amplitude sample, but can be set to a lower normalisation by setting the `master_volume` parameter to a value between `0.` and `1.`.

In [ ]:
soni2.save("../../day_sequence.wav", master_volume=1.0)

In [ ]:
# Plot volume against time evolution
fig, ax1 = plt.subplots()
ax1.set_xlabel('Time [s]')
ax2 = ax1.twinx()
ax1.plot(events2.raw_mapping['time_evo'][0], events2.raw_mapping['volume'][0])
ax1.set_ylabel('Data')
ax1.tick_params(axis ='y')
ax2.set_ylabel('Volume')
ax2.tick_params(axis ='y')

In [ ]:
# Make frames for animation showing volume as a function of time. This takes several minutes.
soni.score.length
import warnings
from pathlib import Path
from strauss.animation import Animate
import shutil
import tempfile

here = Path.cwd()

# Define the final target directory
target_dir_name = Path("figure_animations") / "DaySequence"

# Use a temporary directory for all intermediate files
with tempfile.TemporaryDirectory() as temp_dir_str:
    temp_dir = Path(temp_dir_str)
    print(f"Using temporary directory: {temp_dir}")
    print("This will take a few minutes")

    pipe = Animate(temp_dir)
    pipe.register('volume', sonification=soni, stype='animation')
    xp = events2.raw_mapping['time_evo'][0]
    yp = events2.raw_mapping['volume'][0]
    nframe = int(soni.score.length*int(pipe.pars['fps']))
    xf = np.linspace(xp[0], xp[-1], nframe)
    yf = np.interp(xf, xp, yp)
    xp, yp = xf, yf
    for i in range(xp.size)[::1]:
        fig, ax1 = plt.subplots()
        plt.title("Day Sequence")
        ax1.set_xlabel('Time [s]')
        ax2 = ax1.twinx()
        ax1.plot(xp, yp)
        ax1.set_ylabel('Data')
        ax1.tick_params(axis ='y')
        ax1.axvline(xp[i], ls ='--', c='C0',lw=1.5, alpha=0.55)
        ax1.axhline(yp[i], ls ='--', c='C0',lw=1.5, alpha=0.55)
        ax2.set_ylabel('Volume')
        ax2.tick_params(axis ='y')
        plt.savefig(pipe.frames["volume"].parent / f'frame_{i:05d}.png', dpi=120)
        plt.close()
    print(f"Volume frames created!")
    pipe.render()
    temp_final_mp4 = temp_dir / "final.mp4"
    target_dir_name.mkdir(parents=True, exist_ok=True)
    final_target_path = target_dir_name / temp_final_mp4.name

    if temp_final_mp4.exists():
        shutil.copy(temp_final_mp4, final_target_path)
        print(f"\nFinal animation copied to: {final_target_path}")
    else:
        warnings.warn(f"Could not find {temp_final_mp4} after rendering.")

from IPython.display import Video
Video(f"{here}/figure_animations/DaySequence/final.mp4", embed=True, width=960)